# E2 — Context Definitions → Gold Labels
Turn the 51 raw ExtraSensory labels into the **fixed 3-field schema** (plan §2):
`location / activity / companion`. Multi-label → one value per field by priority; then
mark which samples are **scoreable** (field not all-NaN). Saves the gold table for E3/E4.

In [1]:
import typing
import pandas as pd
import ee_common as ee

ee.ensure_output_dirs()
print("ContextLabel schema (LLM/ML may only fill these):")
for f, T in [("location", ee.LocationT), ("activity", ee.ActivityT), ("companion", ee.CompanionT)]:
    print(f"  {f:9s}: {list(typing.get_args(T))}")

ContextLabel schema (LLM/ML may only fill these):
  location : ['home', 'work', 'gym', 'beach', 'restaurant', 'transit', 'outdoors', 'unknown']
  activity : ['sitting', 'standing', 'walking', 'running', 'cycling', 'exercise', 'sleeping', 'eating', 'unknown']
  companion: ['alone', 'with_friends', 'with_family', 'with_coworkers', 'unknown']


In [2]:
# LABEL_MAP (priority = order). First matching value with a source label == 1 wins.
for name, m in [("LOCATION_MAP", ee.LOCATION_MAP),
                ("ACTIVITY_MAP", ee.ACTIVITY_MAP),
                ("COMPANION_MAP", ee.COMPANION_MAP)]:
    print(name)
    for v, cols in m.items():
        print(f"   {v:14s} <- {', '.join(c.replace('label:', '') for c in cols)}")
print("companion else -> 'alone' (inferred);  activity 'lying' -> 'unknown' (schema fold)")

LOCATION_MAP
   gym            <- AT_THE_GYM
   beach          <- LOC_beach
   home           <- LOC_home
   work           <- LOC_main_workplace, AT_SCHOOL, IN_CLASS, IN_A_MEETING, LAB_WORK
   restaurant     <- FIX_restaurant, AT_A_BAR, AT_A_PARTY
   transit        <- IN_A_CAR, ON_A_BUS, ELEVATOR, DRIVE_-_I_M_THE_DRIVER, DRIVE_-_I_M_A_PASSENGER
   outdoors       <- OR_outside
ACTIVITY_MAP
   sleeping       <- SLEEPING
   running        <- FIX_running
   cycling        <- BICYCLING
   exercise       <- OR_exercise, STAIRS_-_GOING_UP, STAIRS_-_GOING_DOWN
   walking        <- FIX_walking, STROLLING
   eating         <- EATING
   sitting        <- SITTING
   standing       <- OR_standing
   lying          <- LYING_DOWN
COMPANION_MAP
   with_friends   <- WITH_FRIENDS
   with_coworkers <- WITH_CO-WORKERS
companion else -> 'alone' (inferred);  activity 'lying' -> 'unknown' (schema fold)


## 1. Derive gold labels for all users

In [3]:
df = ee.load_all()
gold = ee.derive_gold_labels(df)
cols = ["uuid", "timestamp",
        "gold_location", "gold_activity", "gold_companion",
        "elig_location", "elig_activity", "elig_companion"]
gold[cols].head()

,uuid,timestamp,gold_location,gold_activity,gold_companion,elig_location,elig_activity,elig_companion
0,00EABED2-271D-49D8-B599-1D4A09240601,1444079161,work,sitting,with_coworkers,True,True,True
1,00EABED2-271D-49D8-B599-1D4A09240601,1444079221,work,sitting,with_coworkers,True,True,True
2,00EABED2-271D-49D8-B599-1D4A09240601,1444079281,work,sitting,with_coworkers,True,True,True
3,00EABED2-271D-49D8-B599-1D4A09240601,1444079341,work,sitting,with_coworkers,True,True,True
4,00EABED2-271D-49D8-B599-1D4A09240601,1444079431,work,sitting,with_coworkers,True,True,True


## 2. Distribution per field (scoreable samples only)

In [4]:
for f in ee.FIELDS:
    elig = gold[f"elig_{f}"]
    vc = gold.loc[elig, f"gold_{f}"].value_counts()
    print(f"=== {f}  |  scoreable {elig.mean():.1%} of samples "
          f"({elig.sum():,} / {len(gold):,}) ===")
    print((vc / vc.sum()).map(lambda x: f"{x:6.1%}").to_string(), "\n")

=== location  |  scoreable 98.6% of samples (372,168 / 377,346) ===
gold_location
home           41.1%
unknown        33.6%
work           18.4%
transit         3.3%
outdoors        2.2%
restaurant      0.9%
gym             0.3%
beach           0.2% 

=== activity  |  scoreable 81.7% of samples (308,320 / 377,346) ===
gold_activity
sitting      39.0%
sleeping     26.9%
standing     11.2%
unknown       7.7%
walking       7.0%
eating        5.3%
cycling       1.6%
exercise      0.9%
running       0.4% 

=== companion  |  scoreable 51.1% of samples (192,771 / 377,346) ===
gold_companion
alone              84.4%
with_friends       12.8%
with_coworkers      2.8% 



## 3. Rare classes & known confusions
Strong class imbalance → **report macro-F1 + balanced accuracy** (never raw accuracy).
Rare location classes (gym / beach) and outdoors↔gym↔beach are the confusions to watch in E4.

In [5]:
loc = gold.loc[gold["elig_location"], "gold_location"].value_counts()
print("location class counts (scoreable):")
print(loc.to_string())
print("\nrarest location classes:", list(loc.tail(3).index))

location class counts (scoreable):
gold_location
home          152777
unknown       125131
work           68580
transit        12303
outdoors        8166
restaurant      3475
gym             1151
beach            585

rarest location classes: ['restaurant', 'gym', 'beach']


## 4. Save gold table for E3/E4

In [6]:
gold_out = gold[cols].copy()
p = ee.DATA_DIR / "e2_gold_labels.parquet"
gold_out.to_parquet(p, index=False)
# human-readable distribution for the results folder
dist = []
for f in ee.FIELDS:
    elig = gold[f"elig_{f}"]
    vc = gold.loc[elig, f"gold_{f}"].value_counts()
    for cls, n in vc.items():
        dist.append({"field": f, "class": cls, "n": int(n),
                     "share": round(n / vc.sum(), 4)})
pd.DataFrame(dist).to_csv(ee.RESULTS_DIR / "e2_gold_distribution.csv", index=False)
print("saved:")
print("  ", p, f"({len(gold_out):,} rows)")
print("  ", ee.RESULTS_DIR / "e2_gold_distribution.csv")

saved:
   C:\Project\Apple Health Data\data\enrichment_experiment\e2_gold_labels.parquet (377,346 rows)
   C:\Project\Apple Health Data\results\enrichment_experiment\e2_gold_distribution.csv


**E2 done.** Gold labels + eligibility masks derived and saved.

**Open decision (flagged):** `LYING_DOWN` currently folds to `activity='unknown'` because the
fixed schema (plan §2) has no `lying`. If the professor wants `lying` as a first-class activity,
add it to `ee_common.ActivityT` and set `ACTIVITY_TO_SCHEMA = {}`.

Next: **E3** — the ML control and the Gemini LLM enricher, same `enrich(row) -> ContextLabel`
interface, scored in **E4**.